# Main Case: Replicating Lee & Wooldridge (2026), Section 7.2

ECO 4400 &middot; Applied Research & Case Analytics &middot; Journal Replication, Stage 2

A law firm has retained Elite Economic Advisors -- your firm -- to check whether Cheng & Hoekstra's finding holds up before they rely on it. You're the analyst on the engagement.

The warm-up simplified castle-doctrine adoption to a clean 2x2, comparing one treated cohort against never-adopters over a single pre and post year. Real adoption wasn't nearly that clean. Twenty-one states adopted castle-doctrine laws, in five different years between 2005 and 2009.

1. **The Bridge**. What happens if you run the warm-up's regression on the full, staggered-timing panel, without any special correction?
2. **The Lee & Wooldridge fix**. A tiny worked example first, to build the intuition by hand. Then their actual method, run through a published Python package built for exactly this paper.
3. **Eyeball comparison**. How your numbers stack up against Cheng & Hoekstra's (2013) own published results.

**Required reading:** Cheng & Hoekstra (2013), *"Does Strengthening Self-Defense Law Deter Crime or Escalate Violence?"* (the original castle-doctrine paper) and Lee & Wooldridge (2026), *"Simple Approaches to Inference with Difference-in-Differences Estimators with Small Cross-Sectional Sample Sizes,"* Section 7.2.

**This notebook is a scaffold, not a finished script.** Each section tells you what to compute; you write the code yourself in the empty cell underneath. Work top to bottom -- later cells reuse variables you build in earlier ones, so match the variable names the instructions ask for.

## Setup

Same source as the warm-up: `castle.dta`, loaded directly from a public URL. This time the panel stays whole, covering every adoption cohort across the full 2000-2010 window instead of just 2006 and just two years.

In [ ]:
# Your code here
# - import pandas as pd and statsmodels.formula.api as smf
# - load castle.dta into a DataFrame called df (same URL as the warm-up)
# - look at df[["sid", "year", "effyear", "cdl", "l_homicide", "popwt"]].head()


## Part 1: The Bridge

`castle.dta` already ships with `cdl`, Cheng & Hoekstra's own treatment variable. It measures the proportion of year `t` that state `i` had an effective Castle Doctrine law in place: zero before adoption, one for a full year under the law, a fraction during the adoption year itself. That variable lets us run the same twoway-fixed-effects regression as the warm-up's Method 4, this time across the whole panel.

In [ ]:
# Your code here
# - regress l_homicide on cdl, with state and year fixed effects
#   (C(sid) + C(year)), clustered by state; save as bridge_unw
# - refit the same regression weighted by popwt (smf.wls instead of
#   smf.ols); save as bridge_w
# - print the cdl coefficient and standard error for both


### Why doesn't this fully solve the staggered-timing problem?

The regression above still runs and gives a sensible-looking number (close to the warm-up's 0.108) -- so what's the issue? With staggered adoption, a standard twoway-fixed-effects regression implicitly uses *already-treated* states as part of the comparison group for *later*-treated states in some of its underlying 2x2 comparisons. Goodman-Bacon (2021) shows these "forbidden comparisons" can bias the pooled coefficient, even though nothing in the regression looks wrong on its face. `castle.dta` is one of the standard textbook examples used to illustrate exactly this problem.

This is the motivation for Lee & Wooldridge's approach in Part 2: instead of pooling every cohort into one regression, transform the panel first so that each unit contributes exactly one clean, appropriately-defined comparison.

## Part 2: The Lee & Wooldridge Small-N Approach

The paper's fix is to collapse each state's entire multi-year history into a single number: how much did this state's homicide rate change, relative to its own pre-treatment self? Once every state is one number, we have a simple cross-section instead of a panel, and an ordinary regression on it sidesteps the staggered-timing and small-cluster-count problems at once.

Building that intuition by hand first is worth the time, since the control-state calculation is the one part of this method that isn't obvious on sight. Below: a tiny four-state example you can check with a calculator, then the same idea applied to all 21 treated states using `lwdid`, a Python package built specifically for this paper.

### A tiny example first

Four made-up states, small enough to check by hand -- not real data. Three treated (two adopting in 2006, one in 2008), one never treated. Log homicide rate values are invented for clarity, not pulled from `castle.dta`.

**State A** (adopted 2006): '04: 1.80, '05: 1.80, '06: 2.00, '07: 2.20

Pre mean = (1.80 + 1.80) / 2 = 1.80
Post mean = (2.00 + 2.20) / 2 = 2.10
&Delta;y_A = 2.10 &minus; 1.80 = **0.30**

The same arithmetic, for the rest:

| State | Adopted (g) | Pre mean | Post mean | &Delta;y |
|---|---|---|---|---|
| A | 2006 | 1.80 | 2.10 | **0.30** |
| B | 2006 | 1.50 | 1.75 | **0.25** |
| C | 2008 | 1.50 | 1.65 | **0.15** |

Cohort sizes: 2006 has 2 states (A, B), 2008 has 1 (C). Total treated = 3. **omega_2006 = 2/3, omega_2008 = 1/3.**

**State N** (never treated): '04: 1.20, '05: 1.25, '06: 1.30, '07: 1.30, '08: 1.33, '09: 1.33

State N has no adoption year of its own, so it tries the split at every cohort's year, then blends the results using the weights above.

| Hypothetical split | Pre mean | Post mean | Diff | Weight | Weighted term |
|---|---|---|---|---|---|
| As if g=2006 | 1.225 | 1.30 | 0.075 | 2/3 | 0.0500 |
| As if g=2008 | 1.30 | 1.33 | 0.030 | 1/3 | 0.0100 |

Blended &Delta;y_N = 0.0500 + 0.0100 = **0.0600**

The 2006 split dominates the blend, 0.0500 to 0.0100, because more states actually experienced it. That's the whole mechanism: bigger cohorts get more say in what a control state's counterfactual should look like.

### What this has in common with the warm-up, and with last time

Every state is now one number: A = 0.30, B = 0.25, C = 0.15, N = 0.0600. From here, finding the effect is exactly the same move as the warm-up: average the treated numbers, average the control numbers, subtract.

mean(treated) = (0.30 + 0.25 + 0.15) / 3 = 0.2333
tau = 0.2333 &minus; 0.0600 = 0.1733

All the machinery above -- cohorts, weights, the blend -- exists to get every state down to one honest number. Once that's done, the estimate itself is four averages and a subtraction, same as Stage 1.

One more connection worth making: population weighting (the Bridge regression, and Cheng & Hoekstra's own tables) gives more say to states whose numbers are more reliable. Cohort weighting does the same math move for a different reason: the 2006 split gets more say because more states actually experienced it. Same idea, applied to two different things.

### Now do it for real, with `lwdid`

The toy example above used four states and two cohorts. The real data has 21 treated states across five cohorts and 29 never-treated controls -- same idea, more bookkeeping. Rather than hand-build that loop, use `lwdid`, a Python package built specifically to implement this paper (`pip install lwdid==0.2.3`, then `import lwdid`).

**Which columns to point it at:**
- `data` -- your DataFrame, `df`.
- `y` -- the outcome column, `"l_homicide"`.
- `ivar` -- the unit identifier, `"sid"`.
- `tvar` -- the time variable, `"year"`.
- `gvar` -- each state's first-treatment year, `"effyear"`. Never-treated states are missing values in this column, which is exactly how `lwdid` expects them. `castle.dta` already comes this way, so there's no need to build a `first_treat` column yourself.

**Which method to use:**
- `rolling="demean"` matches Tables 1 and 2's pre/post-mean-then-subtract move, just automated.
- `control_group="never_treated"` matches the toy example: only never-adopters count as controls.
- `estimator="ra"` is the plain OLS regression behind "average treated minus average control."
- `aggregate="overall"` gives one pooled number, `tau_omega`, instead of a separate estimate per cohort.
- `vce=None` gives classical standard errors -- the paper's actual contribution, exact inference that holds up even with very few treated or control units.

In [ ]:
# Your code here
# - !pip install lwdid==0.2.3, then import lwdid
#   tip: the leading "!" runs a terminal command from inside the notebook.
#   lwdid.lwdid(...) takes each input as a name=value keyword argument, so you
#   can list data=, y=, ivar=, etc. in any order -- match them to the
#   parameter list described above.
# - call lwdid.lwdid(...) using the parameters described above; save the
#   result as "result"
# - extract the numbers you'll need below:
#     tau = result.att_overall
#     se_classical = result.se_overall
#     t_classical = result.t_stat_overall
# - print tau and se_classical, and compare to the paper: tau_omega = 0.092,
#   SE = 0.057 (t = 1.61)


## Part 3: Eyeball Comparison with Cheng & Hoekstra (2013)

None of these numbers should match exactly -- they're different specifications on different slices of the same idea. What matters is whether they agree on sign and land in the same neighborhood.

In [ ]:
print("Estimate                                                      Value")
print("-" * 70)
print("Warm-up: single-cohort 2x2 (2006 vs. never-treated, 05->06)   0.108")
print("Bridge: full-panel TWFE, unweighted (this notebook)           0.088")
print("Bridge: full-panel TWFE, population-weighted (this notebook)  0.080")
print(f"Lee & Wooldridge: cohort-weighted small-N (this notebook)      {tau:.3f}")
print("Cheng & Hoekstra Table 5, published range across columns:")
print("  Weighted OLS:    0.080 to 0.100")
print("  Unweighted OLS:  0.058 to 0.088")
print("  Preferred spec (Col 3, region x year FE + controls, weighted): 0.0937")
print("-" * 70)

### A note on the two different "weights" in this exercise

It's easy to conflate these, since they're unrelated ideas that happen to share a name:

- **Cohort weights (`omega_g`)**, used in Part 2, are Lee & Wooldridge's actual method: how to blend a control state's pre/post comparison across every possible treatment-year split. This is the thing being replicated.
- **Population weights (`popwt`)**, used in Part 1's bridge regression and throughout Cheng & Hoekstra's own Table 5, are a standard econometric choice: give larger states more influence in the regression. It has nothing to do with staggered timing or cohorts.

Lee & Wooldridge's method in Part 2 does not use population weights at all.

## Your answer

In 4&ndash;5 sentences, written the way you'd actually explain this to the partner on the engagement: how does your Bridge estimate compare to the warm-up's simplified 2x2? Does your Lee & Wooldridge estimate land closer to Cheng & Hoekstra's published numbers, and why might that be? What does the gap between the Bridge and the Lee & Wooldridge estimate tell you about the risk of running a standard regression on staggered-timing data without correcting for it?

*Double-click this cell to write your answer here.*